# Trabajo Práctico 2 - Grupo 02

### Modelo Bayes Naive

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

El objetivo es establecer un baseline con Multinomial Bayes Naive sobre dos representaciones de texto, Bag of Words y TF-IDF, con búsqueda de hiperparámetros.

NB es el modelo canónico para clasificación de texctos, es rápido de entrenar y transparente, ya que los pesos son log-probabilidades por palabra y clase.

Cualquier modelo mas sofisticado debe superar el F1-macro de NB, si no lo hace hay un bug.

## Importación e instalación de dependencias


In [ ]:
!pip install -q spacy
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 90.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

In [ ]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_bow, SEED
from common.evaluation import evaluate
from common.io_utils import save_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [ ]:
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [ ]:
print("Preprocesando")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Preprocesando
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega N4: Bayes Naive + BoW con priors uniformes

Los priors son la probabilidad que el modelo le asigna a cada clase antes de ver el texto.

Naive Bayes clasifica calculando, para cada clase, la probabilidad de que un texto pertenezca a ella. Esa probabilidad tiene dos partes que se multiplican:

P(clase | texto) = P(texto | clase) × P(clase)

P(texto | clase) es la probabilidad de ver esas palabras dado que la reseña es, por ejemplo, negativa.

P(clase) es el prior, qué tan probable es esa clase en general, sin importar el texto.

Con priors por defecto (class_prior=None), los aprende del data: negativa 40%, neutra 20%, positiva 40%. Entonces antes de leer una sola palabra, el modelo ya cree que una reseña tiene el doble de chances de ser negativa o positiva que neutra.

In [ ]:
# Agrupa vectorizador y modelo en un solo objeto
pipe = Pipeline([
    ("bow", make_bow()),
    ("nb",    MultinomialNB(alpha=1.0, class_prior=[1/3, 1/3, 1/3]))])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_val)
evaluate("nb_bow_v4", y_val, y_pred, hyperparams={"alpha": 1.0, "vectorizer": "BoW", "class_prior": "uniform"})

# Reentrenar en train completo y generar submission
pipe.fit(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val]))
save_model(pipe, "nb_bow_v4")
make_submission(pipe, pd.DataFrame({"id": test["id"], "text": X_test}), "submission_nb_bow_v4.csv");


=== nb_bow_v4 ===
Hiperparámetros: {'alpha': 1.0, 'vectorizer': 'BoW', 'class_prior': 'uniform'}

F1-macro:  0.6533
Precision: 0.6577
Recall:    0.6559
Accuracy:  0.6875

              precision    recall  f1-score   support

    negativa     0.7770    0.7071    0.7404      4080
      neutra     0.3793    0.4975    0.4304      2040
    positiva     0.8168    0.7630    0.7890      4080

    accuracy                         0.6875     10200
   macro avg     0.6577    0.6559    0.6533     10200
weighted avg     0.7134    0.6875    0.6979     10200

Matriz de confusión (filas=real, cols=predicho):
          negativa  neutra  positiva
negativa      2885     973       222
neutra         549    1015       476
positiva       279     688      3113
Modelo guardado → models/nb_bow_v4.joblib
Predicción guardada → submissions/submission_nb_bow_v4.csv  (8500 predicciones)
Distribución: clase 0: 36.8%, clase 1: 25.6%, clase 2: 37.7%
